# Qwen2.5-VL Step-DPO Fine-Tuning (Single T4 on Kaggle)

This notebook trains a QLoRA adapter on the extracted Step-DPO pairs to align the model's reasoning format, based on an expert review.

In [ ]:
!pip install -qU transformers accelerate peft bitsandbytes trl datasets
!pip install -q qwen-vl-utils

In [ ]:
import logging
import datasets
import transformers
datasets.logging.set_verbosity_info()
transformers.logging.set_verbosity_info()

import json
import torch
from datasets import Dataset
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig

# 1. Load Dataset
data_path = "experiments/001_500_reasoning/data/step_dpo_pairs.jsonl"
with open(data_path, 'r') as f:
    raw_data = [json.loads(line) for line in f]
    
hf_data = {"prompt": [], "chosen": [], "rejected": []}
for item in raw_data:
    prompt_content = [
        {"type": "image", "image": item["image_path"]},
        {"type": "text", "text": "Analyze this chart. Provide step-by-step reasoning and a final answer.\n" + item.get("question", "")}
    ]
    # TRL VLM support expects conversational dictionaries.
    # To avoid "open assistant turn" formatting bugs, we append the prefix directly to chosen and rejected.
    hf_data["prompt"].append([{"role": "user", "content": prompt_content}])
    
    full_chosen = item["prefix"] + item["chosen"] + "\n"
    full_rejected = item["prefix"] + item["rejected"] + "\n"
    
    hf_data["chosen"].append([{"role": "assistant", "content": full_chosen}])
    hf_data["rejected"].append([{"role": "assistant", "content": full_rejected}])

dataset = Dataset.from_dict(hf_data)
print(f"Loaded {len(dataset)} pairs.")

In [ ]:
# 2. Load Model & Processor
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    device_map={"": 0}  # Fit completely on single T4 GPU
)

model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()

# Freeze Vision Tower
if hasattr(model, "visual"):
    model.visual.requires_grad_(False)

processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=512*28*28)
processor.tokenizer.padding_side = 'right'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

In [ ]:
# 3. Configure LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

In [ ]:
# 4. Define DPO Trainer
training_args = DPOConfig(
    output_dir="./dpo_qwen_vl",
    beta=0.1, # KL penalty
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5, # Adjusted from 5e-6 for small dataset
    num_train_epochs=3,
    logging_steps=1, # Verbose logging as requested
    save_steps=10,   # Frequent checkpointing
    gradient_checkpointing=True,
    dataset_num_proc=1,
    remove_unused_columns=False,
    report_to="none",
)

# No custom formatting func needed if TRL version >= 0.12.0 supports conversational VLM data directly.

print("\n=== INITIALIZING DPOTRAINER ===")
print("This step runs dataset.map() to tokenize images and text. It may take 1-3 minutes...")

trainer = DPOTrainer(
    model,
    ref_model=None,
    args=training_args,
    train_dataset=dataset,
    processing_class=processor,
    peft_config=peft_config,
)

print("\n=== DPOTRAINER INITIALIZATION COMPLETE ===")

In [ ]:
# 5. Train and Save
print("\n=== STARTING TRAINING LOOP ===")
trainer.train()
trainer.save_model("qwen_vl_step_dpo_adapter")
print("Training complete and adapter saved.")